In [ ]:
from fastai.torch_basics import *
from fastai.data.all import *
from fastai.text.data import *
from tokenizers import Tokenizer
from datasets import load_dataset

/Users/pengren/ws/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
#| default_exp text.tokenizer


# Text tokenizer

> Encode texts with a Hugging Face tokenizer into a flat cache of token IDs and offsets for each document

In [ ]:
EOS_TOK = "<eos>"
BOS_TOK = "<bos>"
PAD_TOK = "<pad>"

def hf_tokenizer(name='gpt2'):
    "Pretrained Hugging Face tokenizer `name`, with `EOS_TOK`, `BOS_TOK` and `PAD_TOK` added as special tokens"
    tok = Tokenizer.from_pretrained(name)
    tok.add_special_tokens([EOS_TOK, BOS_TOK, PAD_TOK])
    return tok

In [ ]:
tok = hf_tokenizer()
pad_id = tok.token_to_id(PAD_TOK)
tok.get_vocab_size(), pad_id

(50260, 50259)

## Token cache

A cache is a directory holding the IDs, the offsets and the tokenizer that produced them, so it can be decoded later without knowing how it was built. 

In [ ]:
def save_token_cache(path, flat, offsets, tok):
    "Save token IDs, document offsets and their tokenizer to directory `path`"
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    np.save(path/'flat.npy', flat)
    np.save(path/'offsets.npy', offsets)
    tok.save(str(path/'tokenizer.json'))

def load_token_cache(path):
    "Load token IDs, document offsets and their tokenizer from directory `path`"
    path = Path(path)
    return np.load(path/'flat.npy'), np.load(path/'offsets.npy'), Tokenizer.from_file(str(path/'tokenizer.json'))

## Encoding texts

`tokenize_texts` encodes a collection of texts with the fast batch encoder and returns two arrays: `flat`, every document's token IDs, and `offsets`, where document `i` is `flat[offsets[i]:offsets[i+1]]`

In [ ]:
def id_dtype(tok):
    "The smallest type that holds every ID of `tok`"
    return np.uint16 if tok.get_vocab_size(with_added_tokens=True) <= 2**16 else np.uint32

In [ ]:
def tokenize_texts(
    items, # Texts, or the items `get_texts` extracts them from
    tok, # HF `Tokenizer`
    get_texts=None, # Column of texts from Dataset, or function from `items` to the texts
    bs=1024, 
    cache_dir=None,
):
    "Encode texts into flat token IDs and document offsets, where document `i` is `flat[offsets[i]:offsets[i+1]]`"
    if isinstance(get_texts, str): get_texts = itemgetter(get_texts)
    texts = items if get_texts is None else get_texts(items)
    texts = list(texts)
    dtype = id_dtype(tok)
    chunks = [np.asarray(e.ids, dtype=dtype) for i in progress_bar(range(0, len(texts), bs)) for e in tok.encode_batch_fast(texts[i:i+bs])]
    flat = np.concatenate(chunks) if chunks else np.empty(0, dtype=dtype)
    offsets = np.cumsum([0] + [len(c) for c in chunks], dtype=np.int64)
    if cache_dir is not None: save_token_cache(cache_dir, flat, offsets, tok)
    return flat, offsets

In [ ]:
texts = ["The quick brown fox", "jumps over the lazy dog"]
flat, offsets = tokenize_texts(texts, tok)
test_eq(offsets[-1], len(flat))
flat, offsets

 |█████████████████████████████████| 100.00% [1/1 00:00<00:00]

(array([  464,  2068,  7586, 21831,    73,  8142,   625,   262, 16931,
         3290], dtype=uint16),
 array([ 0,  4, 10]))

In [ ]:
flat

array([  464,  2068,  7586, 21831,    73,  8142,   625,   262, 16931,
        3290], dtype=uint16)

In [ ]:
offsets

array([ 0,  4, 10])

When the items are a table, a `Dataset` or a `DataFrame`, `get_texts` names the column to encode. A function from the items to their texts covers every other case.


In [ ]:
df = pd.DataFrame({'id': range(len(texts)), 'text': texts})
flat_df, offsets_df = tokenize_texts(df, tok, get_texts='text')
test_eq(flat_df, flat)
test_eq(offsets_df, offsets)


 |█████████████████████████████████| 100.00% [1/1 00:00<00:00]

`split_docs` splits the flat array back into one array per document.

In [ ]:
def split_docs(flat, offsets):
    "The documents of a token cache, as views into `flat`"
    return L(flat[i:j] for i,j in zip(offsets[:-1], offsets[1:]))

In [ ]:
docs = split_docs(flat, offsets)
test_eq(len(docs), len(texts))
tok.decode(docs[1].tolist())

'jumps over the lazy dog'

With cache, `tokenize_texts` saves its result, and `load_token_cache` brings it back with the tokenizer that produced it.

In [ ]:
with tempfile.TemporaryDirectory() as d:
    tokenize_texts(texts, tok, cache_dir=d)
    flat2, offsets2, tok2 = load_token_cache(d)
test_eq(flat2, flat)
test_eq(offsets2, offsets)
tok2.decode(split_docs(flat2, offsets2)[0].tolist())

 |█████████████████████████████████| 100.00% [1/1 00:00<00:00]

'The quick brown fox'

## Hugging Face datasets

This section shows some examples of working with HF datasets. `get_texts` turns the dataset into the texts to encode: a column name picks that column, and `assemble_wiki103` joins WikiText-103 lines into articles


In [ ]:
_wiki_heading = re.compile(r'^ = [^=].* = \n$')

def assemble_wiki103(ds):
    "Join the lines of WikiText-103 dataset `ds` into one document per ` = Title = ` heading"
    docs, cur = [], []
    for text in ds['text']:
        if _wiki_heading.match(text):
            if cur: docs.append(''.join(cur))
            cur = []
        cur.append(text)
    if cur: docs.append(''.join(cur))
    return [d for d in docs if d.strip()]

`tokenize_dataset` does this for a Hugging Face dataset. The first call downloads the dataset, encodes it and saves the cache; later calls load the cache and return its tokenizer. The default cache path is keyed by dataset, subset, split and tokenizer name, so each tokenizer gets its own cache. 

In [ ]:
@delegates(tokenize_texts)
def tokenize_dataset(
    name, # Hugging Face dataset
    get_texts, # Column of the dataset, or function from it to its texts
    subset=None,
    split='train',
    tok='gpt2', # Tokenizer name for `hf_tokenizer`
    cache_dir=None, # Cache directory; `fastai_path('data')/'token_cache'/name/subset/split/tok` if None
    **kwargs
):
    "Token cache for a Hugging Face dataset, loaded from `cache_dir` when it exists, otherwise encoded and saved there first"
    if cache_dir is None: cache_dir = fastai_path('data')/'token_cache'/name/(subset or 'default')/split/tok
    if not (Path(cache_dir)/'tokenizer.json').exists():
        ds = load_dataset(name, subset, split=split)
        tokenize_texts(ds, hf_tokenizer(tok), get_texts=get_texts, cache_dir=cache_dir, **kwargs)
    return load_token_cache(cache_dir)


In [ ]:
#| eval: false
flat_wiki, offsets_wiki, tok_wiki = tokenize_dataset('Salesforce/wikitext', assemble_wiki103, 'wikitext-103-raw-v1', split='validation')
len(offsets_wiki)-1, len(flat_wiki)


KeyboardInterrupt: 

In [ ]:
#| eval: false
sorted(p.name for p in (fastai_path('data')/'token_cache/Salesforce/wikitext/wikitext-103-raw-v1/validation/gpt2').iterdir())

['flat.npy', 'offsets.npy', 'tokenizer.json']

In [ ]:
#| eval: false
tok_wiki.decode(split_docs(flat_wiki, offsets_wiki)[0].tolist())[:80]


' = Homarus gammarus = \n Homarus gammarus , known as the European lobster or comm'

Simple English Wikipedia from `answerdotai/simplewiki` is already one article per item, in its `md` column.


In [ ]:
#| eval: false
flat_sw, offsets_sw, _ = tokenize_dataset('answerdotai/simplewiki', 'md', 'articles', split='train[:500]')
len(offsets_sw)-1, len(flat_sw)


(500, 752440)

In [ ]:
#| eval: false
tok.decode(split_docs(flat_sw, offsets_sw)[0].tolist())[:80]


'**April** (Apr.) is the fourth month of the year in the Julian and Gregorian cal'

## Text files

`tokenize_folder` is `tokenize_dataset` for a folder of text files. It lists the files under `path` in sorted order, encodes their contents on the first call, and loads the cache afterwards. The documents come back in the same order as the files, so each file can be looked up by its path. A cache whose document count differs from the file count is refused, since the files must have changed.

In [ ]:
tokenize_dataset??

````python
@delegates(tokenize_texts)
def tokenize_dataset(
    name, # Hugging Face dataset
    get_texts, # Column of the dataset, or function from it to its texts
    subset=None,
    split='train',
    tok='gpt2', # Tokenizer name for `hf_tokenizer`
    cache_dir=None, # Cache directory; `fastai_path('data')/'token_cache'/name/subset/split/tok` if None
    **kwargs
):
    "Token cache for a Hugging Face dataset, loaded from `cache_dir` when it exists, otherwise encoded and saved there first"
    if cache_dir is None: cache_dir = fastai_path('data')/'token_cache'/name/(subset or 'default')/split/tok
    if not (Path(cache_dir)/'tokenizer.json').exists():
        ds = load_dataset(name, subset, split=split)
        tokenize_texts(ds, hf_tokenizer(tok), get_texts=get_texts, cache_dir=cache_dir, **kwargs)
    return load_token_cache(cache_dir)
````

**File:** `~/aai-ws/fastai/nbs/<ipython-input-16-b2a3742744b6>`; line: 1

In [ ]:
@delegates(tokenize_texts)
def tokenize_folder(
    path, # Folder of text files
    tok='gpt2', # Tokenizer name for `hf_tokenizer`
    cache_dir=None, # Cache directory; `fastai_path('data')/'token_cache'/path.name/tok` if None
    **kwargs
):
    "Every text file under `path`, their token cache, and its tokenizer; the cache is built on the first call"
    path = Path(path)
    files = get_text_files(path).sorted()
    if cache_dir is None: cache_dir = fastai_path('data')/'token_cache'/path.name/tok
    if not (Path(cache_dir)/'tokenizer.json').exists():
        tokenize_texts(files, hf_tokenizer(tok), get_texts=lambda fs: [f.read_text(encoding='utf-8') for f in fs], cache_dir=cache_dir, **kwargs)
    flat, offsets, tok = load_token_cache(cache_dir)
    docs = split_docs(flat, offsets)
    if len(docs) != len(files): raise ValueError(f'{cache_dir} holds {len(docs)} documents for {len(files)} files: delete it to rebuild')
    return files, docs, tok

In [ ]:
#| eval: false
path = untar_data(URLs.IMDB)
files, imdb_docs, tok_imdb = tokenize_folder(path)
len(files), files[0].relative_to(path), tok_imdb.decode(imdb_docs[0][:30].tolist())

(100002,
 Path('test/neg/0_2.txt'),
 'Once again Mr. Costner has dragged out a movie for far longer than necessary. Aside from the terrific sea rescue sequences, of which there are very')

## Transforms

`HfTokenizer` turns a cached document, or a raw string, into a `TensorText` with `<bos>` and `<eos>` around it, and decodes a `TensorText` back to a string. Its `vocab` and `pad_id` are what `language_model_learner` and `text_classifier_learner` read from `dls`, through the pipeline. `get_lengths` gives the length of each document once the boundary tokens are added, which `LMDataLoader` and `SortedDL` ask for the same way. It returns `None` for items that are neither arrays nor strings, such as the `(doc, label)` pairs of a classifier, and the loader then measures them itself. `Reverse` flips a sequence in both directions. Placed after `HfTokenizer`, it is all a backward language model needs: the loaders, the model and the classifier stay the same.


In [ ]:
BOS_TOK, EOS_TOK, PAD_TOK

('<bos>', '<eos>', '<pad>')

In [ ]:
class HfTokenizer(Transform):
    "Encode texts or cached token IDs with `tok`, adding `BOS_TOK` and `EOS_TOK`, and decode them back to text"
    input_types = (str, np.ndarray)
    def __init__(self, tok):
        self.tok = tok
        self.bos,self.eos,self.pad_id = (tok.token_to_id(t) for t in (BOS_TOK, EOS_TOK, PAD_TOK))
        self.vocab = [tok.id_to_token(i) for i in range(tok.get_vocab_size())]
    def _add_boundaries(self, ids): return TensorText([self.bos] + ids + [self.eos])
    def _len(self, o): return len(o) if isinstance(o, np.ndarray) else len(self.tok.encode(o).ids)
    def get_lengths(self, items):
        items = L(items)
        return L(self._len(o)+2 for o in items) if isinstance(items[0], (np.ndarray, str)) else None
    def encodes(self, s:str): return self._add_boundaries(self.tok.encode(s).ids)
    def encodes(self, s:np.ndarray): return self._add_boundaries(s.tolist())
    def decodes(self, t:TensorText): return TitledStr(self.tok.decode(t.tolist(), skip_special_tokens=False))

In [ ]:
tfm = HfTokenizer(tok)
t = tfm(docs[0])
test_eq(tfm.get_lengths(texts), tfm.get_lengths(docs))
t, tfm.decode(t), tfm.get_lengths(docs)

(TensorText([50258,   464,  2068,  7586, 21831, 50257]),
 '<bos>The quick brown fox<eos>',
 [6, 8])

A raw string is encoded first, then given the same boundaries, so a text and its cached document give the same `TensorText`.

In [ ]:
test_eq(tfm(texts[0]), t)
tfm(texts[1]), tfm.decode(tfm(texts[1]))

(TensorText([50258,    73,  8142,   625,   262, 16931,  3290, 50257]),
 '<bos>jumps over the lazy dog<eos>')

In [ ]:
class Reverse(Transform):
    def encodes(self, o): return o.flip(0)
    def decodes(self, o): return o.flip(0)


In [ ]:
pipe = Pipeline([tfm, Reverse()])
test_eq(pipe(docs[0]), t.flip(0))
pipe.decode(pipe(docs[0]))


'<bos>The quick brown fox<eos>'

## DataLoaders

`HfTextBlock` is this module's `TextBlock`. The pipeline is `tok_tfm` alone, with `Reverse` after it for a backward model, because the tokenizer already produces ids and there is no `Numericalize` step. Padding for classification uses the tokenizer's `pad_id`.

In [ ]:
class HfTextBlock(TransformBlock):
    "A `TransformBlock` for documents that `tok_tfm` encodes straight to token ids"
    def __init__(self, tok_tfm, is_lm=False, seq_len=72, backwards=False):
        type_tfms = [tok_tfm] + ([Reverse()] if backwards else [])
        super().__init__(type_tfms=type_tfms, dl_type=LMDataLoader if is_lm else SortedDL,
                         dls_kwargs={'seq_len': seq_len} if is_lm else {'before_batch': Pad_Chunk(seq_len=seq_len, pad_idx=tok_tfm.pad_id)})

In [ ]:
blk = HfTextBlock(tfm, is_lm=True, seq_len=3)
dls = DataBlock(blocks=blk, splitter=RandomSplitter(0.5, seed=42)).dataloaders(docs, bs=1)
x, y = dls.one_batch()
x.shape, len(dls.vocab), dls.pad_id

(torch.Size([1, 3]), 50260, 50259)

`TextDataLoadersNew` is the `TextDataLoaders` of this module, to take that name once the migration is complete. `from_dataset` builds `DataLoaders` from the token cache of a Hugging Face dataset, through `tokenize_dataset`: a language model by default, or a classifier when `get_y` names the label column or computes the labels from the dataset. `from_folder` works as the one on `TextDataLoaders`: the items are the file paths, `GrandparentSplitter` or `RandomSplitter` splits them, and `parent_label` names the class. The only change is that a path is looked up in the cache from `tokenize_folder` instead of being tokenized. The items are the files of the `train` and `valid` folders, or of `folders` when given, which a random split needs since the cache covers every file under `path`.


In [ ]:
class TextDataLoadersNew(DataLoaders):
    "Wrapper around several `DataLoader`s with factory methods for texts encoded by `HfTokenizer`"
    @classmethod
    @delegates(tokenize_dataset)
    def from_dataset(cls, name, get_texts, subset=None, split='train', get_y=None, valid_pct=0.2, seed=None, seq_len=72, backwards=False, bs=64, **kwargs):
        "`DataLoaders` from the token cache of a Hugging Face dataset: a language model, or a classifier when `get_y` gives the labels"
        flat, offsets, tok = tokenize_dataset(name, get_texts, subset, split=split, **kwargs)
        docs = split_docs(flat, offsets)
        is_lm = get_y is None
        blocks = [HfTextBlock(HfTokenizer(tok), is_lm=is_lm, seq_len=seq_len, backwards=backwards)]
        if not is_lm:
            ds = load_dataset(name, subset, split=split)
            labels = ds[get_y] if isinstance(get_y, str) else get_y(ds)
            if len(labels) != len(docs): raise ValueError(f'{len(labels)} labels for {len(docs)} documents: `get_texts` and `get_y` must give one item each per document')
            blocks.append(CategoryBlock)
        dblock = DataBlock(blocks=blocks,
                           get_x=lambda i: docs[i],
                           get_y=None if is_lm else (lambda i: labels[i]),
                           splitter=RandomSplitter(valid_pct, seed=seed))
        return cls.from_dblock(dblock, range_of(docs), bs=bs, seq_len=seq_len)

    @classmethod
    @delegates(DataLoaders.from_dblock)
    def from_folder(cls, path, train='train', valid='valid', valid_pct=None, seed=None, is_lm=False, tok='gpt2', seq_len=72, backwards=False, folders=None, cache_dir=None, **kwargs):
        "Create from imagenet style dataset in `path` with `train` and `valid` subfolders (or provide `valid_pct`)"
        files, docs, tok = tokenize_folder(path, tok, cache_dir=cache_dir)
        doc_of = dict(zip(files, docs))
        blocks = [HfTextBlock(HfTokenizer(tok), is_lm=is_lm, seq_len=seq_len, backwards=backwards)]
        if not is_lm: blocks.append(CategoryBlock)
        if valid_pct is None: splitter,folders = GrandparentSplitter(train_name=train, valid_name=valid),ifnone(folders, [train, valid])
        else: splitter = RandomSplitter(valid_pct, seed=seed)
        dblock = DataBlock(blocks=blocks, get_x=doc_of.__getitem__, get_y=None if is_lm else parent_label, splitter=splitter)
        return cls.from_dblock(dblock, get_text_files(path, folders=folders), path=path, seq_len=seq_len, **kwargs)


In [ ]:
#| eval: false
dls = TextDataLoadersNew.from_dataset('Salesforce/wikitext', assemble_wiki103, 'wikitext-103-raw-v1', split='validation', valid_pct=0.1, seed=42, bs=8)
dls.show_batch(max_n=2)

<bos> = Exit Through the Kwik @-@ E @-@ Mart = 
 " Exit Through the Kwik @-@ E @-@ Mart " is the fifteenth episode of the twenty @-@ third season of the American animated television sitcom The Simpsons . It originally aired on the Fox network in the United States on March 4 , 2012 .
 of classic movies . 
 Highland Park houses a Jimmie Rodgers museum which displays the original guitar of " The Singing Brakeman " and other memorabilia of his life and career , as well as railroad equipment from the steam @-@ engine era . In addition to the museum building itself , there are outside memorials , and a vintage steam locomotive
 = Exit Through the Kwik @-@ E @-@ Mart = 
 " Exit Through the Kwik @-@ E @-@ Mart " is the fifteenth episode of the twenty @-@ third season of the American animated television sitcom The Simpsons . It originally aired on the Fox network in the United States on March 4 , 2012 . In
 classic movies . 
 Highland Park houses a Jimmie Rodgers museum which displays the origi

In [ ]:
#| eval: false
dls = TextDataLoadersNew.from_dataset('stanfordnlp/imdb', 'text', get_y=lambda ds: ds.features['label'].int2str(ds['label']), valid_pct=0.2, seed=42, bs=8)
dls.show_batch(max_n=2)


,text,category
0,"<bos>Match 1: Tag Team Table Match Bubba Ray and Spike Dudley vs Eddie Guerrero and Chris Benoit Bubba Ray and Spike Dudley started things off with a Tag Team Table Match against Eddie Guerrero and Chris Benoit. According to the rules of the match, both opponents have to go through tables in order to get the win. Benoit and Guerrero heated up early on by taking turns hammering first Spike and then Bubba Ray. A German suplex by Benoit to Bubba took the wind out of the Dudley brother. Spike tried to help his brother, but the referee restrained him while Benoit and Guerrero ganged up on him in the corner. With Benoit stomping away on Bubba, Guerrero set up a table outside. Spike dashed into the ring and somersaulted over the top rope onto Guerrero on the outside! After recovering and taking care of Spike, Guerrero slipped a table",pos
1,"<bos>If you've ever been harassed on the Underground by a Christian who says, ""Jesus is the answer. What's the question?"", then perhaps you should thank God if you've never met a Lacanian. Slavoj Zizek, the most evangelical of Lacanians, would surely exchange the word ""Jesus"" in that statement for ""Lacan/Hegel"".<br /><br />Zizek's star burns brightly at the moment, no doubt because we generally view films and pop culture purely as entertainment for our consumption. So it seems impressive when someone - anyone - comes along and says, ""Hang on, films may say something about ourselves.""<br /><br />The ideas Zizek expounds in this film are ""true"" purely because he says so. For example, Zizek explains that three Marx Bros are the ego, superego and id (God knows what happened to Zeppo, or Gummo perhaps they're the sinthome...or is that movies themselves?). This is simply what they are. In Zizek's output,",neg


## IMDb

The IMDb reviews come as text files in `train` and `test` folders, with `pos`, `neg` and `unsup` subfolders. `from_folder` with `valid='test'` gives the classifier loaders over the labelled reviews, and with `is_lm=True` and `valid_pct` the language model loaders over all of them.


In [ ]:
#| eval: false
dls = TextDataLoadersNew.from_folder(path, valid='test', bs=8)
dls.show_batch(max_n=2)

,text,category
0,"<bos>Match 1: Tag Team Table Match Bubba Ray and Spike Dudley vs Eddie Guerrero and Chris Benoit Bubba Ray and Spike Dudley started things off with a Tag Team Table Match against Eddie Guerrero and Chris Benoit. According to the rules of the match, both opponents have to go through tables in order to get the win. Benoit and Guerrero heated up early on by taking turns hammering first Spike and then Bubba Ray. A German suplex by Benoit to Bubba took the wind out of the Dudley brother. Spike tried to help his brother, but the referee restrained him while Benoit and Guerrero ganged up on him in the corner. With Benoit stomping away on Bubba, Guerrero set up a table outside. Spike dashed into the ring and somersaulted over the top rope onto Guerrero on the outside! After recovering and taking care of Spike, Guerrero slipped a table",pos
1,"<bos>By now you've probably heard a bit about the new Disney dub of Miyazaki's classic film, Laputa: Castle In The Sky. During late summer of 1998, Disney released ""Kiki's Delivery Service"" on video which included a preview of the Laputa dub saying it was due out in ""1999"". It's obviously way past that year now, but the dub has been finally completed. And it's not ""Laputa: Castle In The Sky"", just ""Castle In The Sky"" for the dub, since Laputa is not such a nice word in Spanish (even though they use the word Laputa many times throughout the dub). You've also probably heard that world renowned composer, Joe Hisaishi, who scored the movie originally, went back to rescore the excellent music with new arrangements. Laputa came out before My Neighbor Totoro and after Nausicaa of the Valley of the Wind, which began Studio Ghibli and it's long string of",pos


In [ ]:
#| eval: false
dls_lm = TextDataLoadersNew.from_folder(path, is_lm=True, valid_pct=0.1, seed=42, folders=['train', 'test', 'unsup'], bs=64)
len(dls_lm.train_ds), len(dls_lm.valid_ds), len(dls_lm.vocab)

(90000, 10000, 50260)

In [ ]:
#| eval: false
dls = TextDataLoadersNew.from_folder(path, valid='test', bs=8)
dls.show_batch(max_n=2)

,text,category
0,"<bos>Match 1: Tag Team Table Match Bubba Ray and Spike Dudley vs Eddie Guerrero and Chris Benoit Bubba Ray and Spike Dudley started things off with a Tag Team Table Match against Eddie Guerrero and Chris Benoit. According to the rules of the match, both opponents have to go through tables in order to get the win. Benoit and Guerrero heated up early on by taking turns hammering first Spike and then Bubba Ray. A German suplex by Benoit to Bubba took the wind out of the Dudley brother. Spike tried to help his brother, but the referee restrained him while Benoit and Guerrero ganged up on him in the corner. With Benoit stomping away on Bubba, Guerrero set up a table outside. Spike dashed into the ring and somersaulted over the top rope onto Guerrero on the outside! After recovering and taking care of Spike, Guerrero slipped a table",pos
1,"<bos>Elfriede Jelinek, not quite a household name yet, is a winner of the Nobel prize for literature. Her novel spawned a film that won second prize at Cannes and top prizes for the male and female leads. Am I a dinosaur in matters of aesthetic appreciation or has art become so debased that anything goes?<br /><br />'Gobble, gobble' is the favoured orthographic representation in Britain of the bubbling noise made by a turkey. In the film world a turkey is a monumental flop as measured by box office receipts or critical reception. 'Gobble, gobble' and The Piano Teacher are perfect partners.<br /><br />The embarrassing awfulness of this widely praised film cannot be overstated. It begins very badly, as if made to annoy the viewer. Credits interrupt inconsequential scenes for more than 11 minutes. We are introduced to Professor Erika Kohut, apparently the alter ego of the accoladed authoress, a stony",neg


## Export -

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()